# Explainable AI (SHAP) Analysis for TB Recovery Check

This notebook demonstrates how to load the saved SHAP explainers, compare global feature importances, and perform patient-level local explanation visualization.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import shap
import joblib
from pathlib import Path

# Ensure project root is in path
notebook_dir = Path.cwd()
ROOT_DIR = notebook_dir.parent if (notebook_dir / 'src').exists() == False else notebook_dir
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.explain.shap_explainer import load_explainer, load_global_explanation
from src.explain.visualizations import plot_shap_summary, plot_shap_waterfall, plot_shap_force

## 1. Global Importance Comparison across Aims

We load champion models and explainers to compare feature importances.

In [2]:
from src.api.dependencies import get_model

# Load Aim 1 imputed model info
try:
    pipeline_aim1, v_aim1, m_aim1 = get_model("aim1_non_conversion")
    print(f"Aim 1 model: {m_aim1} ({v_aim1})")
    explainer_aim1 = load_explainer("aim1_non_conversion", m_aim1, v_aim1)  # noqa: F821
    global_aim1 = load_global_explanation("aim1_non_conversion", m_aim1, v_aim1)  # noqa: F821
    print(f"Loaded Aim 1 global SHAP with background n={global_aim1['n_background_samples']}")
except Exception as e:
    print(f"Error loading Aim 1: {e}")

Aim 1 model: xgboost (v9)
Loaded Aim 1 global SHAP with background n=174


## 2. Patient Case Study Visualizations

Visualizing individual sample level risk explanations.

In [3]:
# Load sample patient features
data_path = ROOT_DIR / "data" / "cleaned" / "aim1_patients_imputed.csv"  # noqa: F821
if data_path.exists():
    df_patients = pd.read_csv(data_path)  # noqa: F821
    print(f"Loaded dataset shape: {df_patients.shape}")
else:
    print("Dataset path not found, run pipeline retraining first.")

Loaded dataset shape: (218, 57)
